In [ ]:
import ROOT as rt
import csv
import sys
import collections
import os
from collections import OrderedDict
import uproot

import numpy as np

sys.path.append(os.getcwd().replace('scripts', 'lib'))

import tdrstyle
tdrstyle.setTDRStyle()



print(sys.version)

In [ ]:
def readNorm(f_cscCard):
    f = open(f_cscCard,"r")
    Lines = f.readlines()
    for line in Lines:
        l  = line.split()
        if len(l) == 0:continue
        if l[0] == 'rate':
            return float(l[1])

# Load limit trees

Here you will read the `limit` trees and create an array `limits` that stores 6 number for each LLP lifetimes.

The 6 numbers are 
 - -2 sig limit
 - -1 sig limit
 - expected limit
 - +1 sig limit
 - +2 sig limit
 - observerd limit
 
You can read more about t
https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/latest/part3/runningthetool/#output-from-combine



You will also need to divide the limit from limit trees by the norm that you've set when you ran `Combine`

Scale the limit by the signal_norm in the line that has the comment `CHANGE THIS LINE`

In [ ]:
dataCardDir = os.getenv('CMSSW_BASE') + '/src/MDS_CMSDAS/combine/datacards'
limitDir = os.getenv('CMSSW_BASE') + '/src/MDS_CMSDAS/combine/limitTrees'
ctaus = np.array([10, 50, 100, 500, 1000,5000, 10000, 50000, 100000,int(2e5),int(3e5),int(5e5), \
                        int(2e6),int(3e6),int(5e6),int(1e7)])


limitTrees = {}
dataCards = {}
limits = np.ones((len(ctaus), 6))*1000000


# load the data cards and limit tree ROOT files
for ct in ctaus:
    name = 'ggH_HToSSTobbbb_mh125_mx40'
    dataCards[ct] = dataCardDir + '/'+name+'_ctau'+str(ct)+'mm.txt'
    limitTrees[ct] = limitDir + '/'+'/higgsCombine.'+name + '_ctau'+str(ct)+'mm.AsymptoticLimits.mH120.root'
for i,m in enumerate(limitTrees.keys()):
    if not os.path.isfile(dataCards[m]):
        print(dataCards[m])
        continue
    signal_norm = readNorm(dataCards[m])
    if signal_norm == 0:continue
    if len(uproot.open(limitTrees[m]).keys()) == 2:
        T = uproot.open(limitTrees[m])['limit']
        limits[i] = np.array(T['limit'].array())/signal_norm #CHANGE THIS LINE
limits = np.array(limits)

print(ctaus)



### What was filled in above

`dataCardDir` and `limitDir` now point to the two folders the previous steps created (the notebook is run from `src/MDS_CMSDAS/scripts`, so an absolute path built on `$CMSSW_BASE` is used, exactly like in `run_combine.py`):

 - `$CMSSW_BASE/src/MDS_CMSDAS/combine/datacards` - the `ggH_HToSSTobbbb_mh125_mx40_ctau<ct>mm.txt` cards written by `create_datacard.ipynb`. They are needed here because `readNorm()` re-reads the signal yield of bin A (`sigA`, the second token of the `rate` line) from each card, which is the inverse of the scaling that was imposed on Combine.
 - `$CMSSW_BASE/src/MDS_CMSDAS/combine/limitTrees` - the `higgsCombine.<card>.AsymptoticLimits.mH120.root` files that `run_combine.py` moved there. Each file holds a `limit` tree with 6 rows per point: `-2 sigma, -1 sigma, median expected, +1 sigma, +2 sigma, observed`, which is why `limits` has shape `(n_ctau, 6)`.

The line marked `CHANGE THIS LINE`, `limits[i] = np.array(T['limit'].array())/signal_norm`, is the r -> BR(h -> SS) conversion: the datacard scales the signal by the frozen `rateParam norm = 1/sigA`, so the fitted `r` satisfies `BR = r x norm = r/sigA`. Two guards are worth knowing about: a point whose card or tree is missing keeps the sentinel value `100000` (and is dropped later by `cond = limits[:,2] < 1000000`), and the `len(uproot.open(...).keys()) == 2` test skips files where Combine crashed before writing a `limit` tree (a good file has `limit` + `MetaData`). Points with `sigA = 0` are skipped because the limit would be meaningless (division by zero).

# Plotting expected and observed Limit

Here we will create `TGraph` for the expected(`h_exp`), observed (`h_obs`) and +-1
 band (`h_exp1sig`).
 
`h_exp` and `h_exp1sig` are already filled in, please fill in `h_obs` based on the format of `h_exp`.

In [ ]:
opac_1sig = 0.25


leg = rt.TLegend(0.65,0.2,0.95,0.38)



leg.SetTextSize(0.03)
leg.SetBorderSize(0)
leg.SetEntrySeparation(0.01)
leg.SetTextFont(42)
c = rt.TCanvas('c','c', 800, 800)
c.SetRightMargin(0.04)


rt.gStyle.SetOptFit(1011)
ctaus_m = ctaus/1000

cond = limits[:,2] < 1000000


h = rt.TGraph(len(ctaus_m[cond]),ctaus_m[cond], limits[:,2][cond])
h_obs = rt.TGraph(len(ctaus_m[cond]),ctaus_m[cond], limits[:,5][cond])
h_exp1sig = rt.TGraph(2*len(ctaus_m[cond]), np.hstack((ctaus_m[cond], np.flip(ctaus_m[cond]))), np.hstack((limits[:,1][cond], np.flip(limits[:,3][cond]))))

leg.AddEntry(h_obs, "Observed", "LP")
leg.AddEntry(h, "Median Expected", "L")
leg.AddEntry(h_exp1sig, "#pm 1 #sigma Expected", 'F')

h_exp1sig.SetFillColorAlpha(1,opac_1sig)
h.SetLineColor(1)
h_obs.SetLineColor(1)
h_obs.SetMarkerColor(1)


h_obs.SetLineWidth(3)

h.SetLineStyle(2)
h.SetLineWidth(3)
h.GetXaxis().SetTitle('c#tau [m]')
h.GetYaxis().SetTitle('95% CL Limit on BR (h #rightarrow SS)')
h.GetXaxis().SetLimits(0.01,10000.0)
h.GetYaxis().SetRangeUser(5e-4,50)
h.GetYaxis().SetRangeUser(1e-4,2)

h.GetXaxis().SetTitleOffset(1)
h.GetYaxis().SetTitleSize(0.05)
h.GetYaxis().SetTitleOffset(1.5)

h.Draw('LPA')
h_obs.Draw('Lsame')
h_exp1sig.Draw('Fsame')


leg.Draw()
c.SetLogy()
c.SetLogx()
c.SetTicky(1)
c.SetTickx(1)


latex = rt.TLatex()
latex.SetTextSize(0.06)
latex.SetTextFont(42)


tdrstyle.setTDRStyle()
c.Draw()


### About the three `TGraph`s above

`h` uses `limits[:,2]` (median expected), `h_exp1sig` walks the ctau axis forwards with `limits[:,1]` (-1 sigma) and backwards with `limits[:,3]` (+1 sigma) to close the band, and `h_obs` uses `limits[:,5]`, the last row of the `limit` tree, i.e. the observed limit. All three are already divided by `sigA` in the loading cell, so the y axis is a limit on BR(h -> SS) and not on `r`.

The mask `cond = limits[:,2] < 1000000` removes the points that were never filled by Combine; the axes are taken from `h` (drawn first with `LPA`), which is why `h_obs` is drawn with `Lsame`. A `TGraph` carries its own axes, so all the decorating (`SetTitle`, `SetLimits`, `SetRangeUser`) is done on `h`, the graph drawn first with `LPA`, which is the one that defines the frame. `SetRangeUser` is called twice (5e-4..50, then 1e-4..2), so the second call is what you actually see - if your own limits fall outside it, widen that range or the curves will look clipped.

Shape of the result (Discussion 6.1): the curve falls (better sensitivity) from short ctau, reaches its best limit at intermediate decay lengths, then rises again for long lifetimes - the LLP either decays too early to reach the muon system or leaves the detector without decaying at all.

In [ ]:
# Exercise: overlay the observed limits for mS = 15, 40 and 55 GeV on one canvas.
# Prerequisite 1: re-run create_datacard.ipynb with
#     mass = [15, 40, 55]
# and make the sample name in the loading cell follow m, i.e. replace the hard
# coded 'MS-40' by 'MS-'+str(m):
#     fpath[key] = path + 'ggH_HToSSTobbbb_MH-125_MS-'+str(m)+'_ctau-'+str(ct)+'_137000pb_weighted.root'
# Prerequisite 2: run run_combine.py once per mass, i.e. with
#     sample_name = 'ggH_HToSSTobbbb_mh125_mx15'  (then mx40, then mx55)
# the limit trees of all masses end up in the same limitTrees folder.
masses = [15, 40, 55]
colors = {15: 2, 40: 4, 55: 8}

c_x = rt.TCanvas('c_x', 'c_x', 800, 800)
c_x.SetRightMargin(0.04)
leg_x = rt.TLegend(0.55, 0.18, 0.92, 0.42)
leg_x.SetTextSize(0.03)
leg_x.SetBorderSize(0)
leg_x.SetTextFont(42)

graphs = []
for m in masses:
    lim_m = np.ones((len(ctaus), 6))*1000000
    for i, ct in enumerate(ctaus):
        name = 'ggH_HToSSTobbbb_mh125_mx'+str(m)
        card_m = dataCardDir + '/' + name + '_ctau'+str(ct)+'mm.txt'
        tree_m = limitDir + '/higgsCombine.' + name + '_ctau'+str(ct)+'mm.AsymptoticLimits.mH120.root'
        if not os.path.isfile(card_m) or not os.path.isfile(tree_m): continue
        signal_norm = readNorm(card_m)
        if signal_norm == 0: continue
        fin = uproot.open(tree_m)
        if len(fin.keys()) != 2: continue
        lim_m[i] = np.array(fin['limit']['limit'].array())/signal_norm
    cond_m = lim_m[:, 2] < 1000000
    g_obs_m = rt.TGraph(len(ctaus_m[cond_m]), ctaus_m[cond_m], lim_m[:, 5][cond_m])
    g_obs_m.SetLineColor(colors[m])
    g_obs_m.SetMarkerColor(colors[m])
    g_obs_m.SetLineWidth(3)
    leg_x.AddEntry(g_obs_m, 'm_{S} = '+str(m)+' GeV', 'L')
    graphs.append((g_obs_m, cond_m, lim_m))

first = None
for g_obs_m, cond_m, lim_m in graphs:
    if cond_m.sum() == 0: continue
    if first is None:
        g_obs_m.GetXaxis().SetTitle('c#tau [m]')
        g_obs_m.GetYaxis().SetTitle('95% CL Limit on BR (h #rightarrow SS)')
        g_obs_m.GetXaxis().SetLimits(0.01, 10000.0)
        ymin = min(lim[:, 5][c].min() for g, c, lim in graphs if c.sum() > 0)
        ymax = max(lim[:, 5][c].max() for g, c, lim in graphs if c.sum() > 0)
        g_obs_m.GetYaxis().SetRangeUser(ymin/1.5, ymax*1.5)
        g_obs_m.GetXaxis().SetTitleOffset(1)
        g_obs_m.GetYaxis().SetTitleSize(0.05)
        g_obs_m.GetYaxis().SetTitleOffset(1.5)
        g_obs_m.Draw('LPA')
        first = g_obs_m
    else:
        g_obs_m.Draw('Lsame')
leg_x.Draw()
c_x.SetLogy()
c_x.SetLogx()
c_x.SetTicky(1)
c_x.SetTickx(1)
c_x.Draw()

### Exercise: limits for mS = 15, 40 and 55 GeV on one canvas

The cell above reuses `ctaus`, `ctaus_m`, `dataCardDir`, `limitDir` and `readNorm` from the earlier cells, and loops over `masses` instead of the single hard coded `ggH_HToSSTobbbb_mh125_mx40`: for each mass it rebuilds the `(n_ctau, 6)` limit array, converts `r` to BR(h -> SS) by dividing by that card's own `sigA`, and makes one `TGraph` of the observed points (`lim_m[:, 5]`). The first mass with at least one valid point draws the frame (`LPA`) and fixes the x/y ranges from the global minimum and maximum, the others are added with `Lsame`.

Physics of the overlay: at fixed ctau the curve moves with mS because the boost of the LLP changes - in h -> SS each S has momentum p* = sqrt(mH^2/4 - mS^2), so a heavier S is slower and its lab decay length c*tau*beta*gamma shrinks, pushing decays back towards the tracker instead of the muon stations; a very light S is the opposite extreme (highly boosted, so its two b-jets and their hits are collimated, which changes the cluster size and the efficiency of the cut based ID). The three curves are shifted versions of the same inverted umbrella. Overlaying the three curves reproduces the plot on slide 25 of the introduction slides; read the actual numbers off your own y axis at the minimum of each curve, and check that the ordering follows the boost argument above rather than assuming it.